# Load an Existing Chroma Database

This notebook reconnects to the persisted Chroma collection created in the CRUD notebook and reads its contents.

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_ollama import OllamaEmbeddings

## 1. Rebuild the Same Configuration

In [2]:
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

project_root

WindowsPath('g:/sushant/Advanced_RAG/4.Vector_Store')

In [3]:
dotenv_path = project_root / ".env"
load_dotenv(dotenv_path=dotenv_path)

if not os.getenv("GROQ_API_KEY"):
    raise ValueError("Please add your GROQ_API_KEY to the .env file before running this notebook.")

print(f"Loaded environment from: {dotenv_path}")

Loaded environment from: g:\sushant\Advanced_RAG\4.Vector_Store\.env


In [4]:
collection_name = "demo"
persist_directory = project_root / "db" / "chroma_langchain_db"

print(f"Collection name: {collection_name}")
print(f"Persist directory: {persist_directory}")

Collection name: demo
Persist directory: g:\sushant\Advanced_RAG\4.Vector_Store\db\chroma_langchain_db


In [5]:
# Use the same embedding model that was used to build the store.
embeddings = OllamaEmbeddings(model="qwen3-embedding:4b")

vector_store = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,
    persist_directory=str(persist_directory),
)

print("Connected to the existing Chroma collection.")

Connected to the existing Chroma collection.


## 2. Add Small Display Helpers

In [6]:
def preview_text(text, limit=80):
    """Return a short preview for cleaner notebook output."""
    if len(text) <= limit:
        return text
    return text[:limit] + "..."


def print_stored_documents(records):
    """Print stored Chroma records in a readable format."""
    ids = records.get("ids", [])
    documents = records.get("documents", [])
    metadatas = records.get("metadatas", [])

    print(f"Total documents in collection: {len(ids)}")
    print()

    for index, (doc_id, document_text, metadata) in enumerate(zip(ids, documents, metadatas), start=1):
        print(f"{index}. id={doc_id}")
        print(f"   topic={metadata.get('topic')} | doc_number={metadata.get('doc_number')}")
        print(f"   content={preview_text(document_text)}")
        print()

## 3. Fetch and Inspect the Stored Records

In [7]:
stored_records = vector_store.get(include=["embeddings", "metadatas", "documents"])
stored_records.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])

In [8]:
stored_records["embeddings"].shape

(8, 2560)

In [9]:
print_stored_documents(stored_records)

Total documents in collection: 8

1. id=59b7d47b-705a-4a63-9572-0996d6d615c0
   topic=AI | doc_number=1
   content=Artificial intelligence helps machines perform tasks that usually need human rea...

2. id=ea582284-7040-452a-8d76-b9adf8e3ba5d
   topic=AI | doc_number=2
   content=AI systems can analyze patterns in data to support predictions and automation.

3. id=9b9ab79c-178f-4af9-9463-553208a7899f
   topic=AI | doc_number=3
   content=Responsible AI development includes fairness, transparency, and safety checks.

4. id=930ab738-ce69-4023-a767-cbeb395dc25a
   topic=RAG | doc_number=4
   content=RAG improves answer quality by retrieving relevant context before the language m...

5. id=cf8e83f4-828c-45d4-bccb-98962ba4c328
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model ge...

6. id=0bdc7727-0314-4ccd-a255-1c278fed7a71
   topic=RAG | doc_number=6
   content=Vector stores are important in RAG because they make semantic 